# Relatividad general y métodos de inferencia
## GRAVITAS · cuaderno autónomo

Facundo Firmenich · [ORCID](https://orcid.org/0009-0002-6578-3811) · f.firmenich@cedesur.org

Ejecutar todas extrae una carga verificada, instala dependencias y reproduce cálculos, tablas y figuras. No exige GitHub ni Drive; la primera instalación necesita Internet. Los manuscritos y su información suplementaria se entregan por separado y no están incrustados en este cuaderno.

El grafo ofrece tour, play, pausa, fuentes descargables y límites de alcance. Los cálculos Cassini son condicionales a entradas reconstruidas. No hay nueva cota física ni validación instrumental independiente. iPhone/iPad acceden mediante Colab en el navegador, no mediante un ejecutable nativo.

Software archivado v1.11: [10.5281/zenodo.24381485](https://doi.org/10.5281/zenodo.24381485) · DOI general: [10.5281/zenodo.24381484](https://doi.org/10.5281/zenodo.24381484).


DCAE: I = 0.90 equivale a R = 10 y Δ = ln(10) por normalización; no es una barrera física ni una probabilidad. Véase docs/DCAE_NORMALIZATION_ES.md.


In [ ]:
PAYLOAD_SHA256='000a1a69455f1c396370dd62bdf702cbceb150a5a35e04b743306acbeeec5825'
PAYLOAD='...'
from pathlib import Path
import base64,hashlib,zipfile,io,tempfile,subprocess,sys

def extract():
    raw=base64.b64decode(PAYLOAD)
    if hashlib.sha256(raw).hexdigest()!=PAYLOAD_SHA256:raise RuntimeError('Carga alterada / Payload integrity failure')
    work=Path(tempfile.mkdtemp(prefix='GRAVITAS_'))
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for item in archive.infolist():
            if not (work/item.filename).resolve().is_relative_to(work):raise RuntimeError('Ruta insegura / Unsafe archive path')
        archive.extractall(work)
    return work

WORK=extract()
from importlib.metadata import version,PackageNotFoundError
needed=[]
for line in (WORK/'requirements.txt').read_text().splitlines()+['ipywidgets==8.1.7']:
    name,wanted=line.split('==')
    try:
        if version(name)!=wanted:needed.append(line)
    except PackageNotFoundError:needed.append(line)
if needed:subprocess.check_call([sys.executable,'-m','pip','install',*needed])
print('Workspace / Carpeta:',WORK)
print('SHA-256:',PAYLOAD_SHA256)


## 1 · Ejecutar cálculos, tablas y figuras
Incluye los controles de información, Cassini, Gaia, QUMOND, DCAE y la generación del grafo. Los fallos predictivos científicos se conservan aunque termine correctamente el programa.

In [ ]:
subprocess.run([sys.executable,str(WORK/'run.py'),'all'],cwd=WORK,check=True)

## 2 · Explorar la investigación

In [ ]:
from IPython.display import HTML, display
import html
graph=(WORK/'ui/epistemic/epistemic_graph_es.html').read_text()
display(HTML('<iframe style="width:100%;height:1100px;border:0" srcdoc="'+html.escape(graph,quote=True)+'"></iframe>'))

## 3 · Reproducir por bloques
Elegí el modo y pulsá el botón. No se ejecuta código arbitrario desde el selector.

In [ ]:
import ipywidgets as widgets
selector=widgets.Dropdown(options=[('FractalGo: topología', 'topology'), ('Fermi-LAT', 'fermi'), ('Estabilidad posterior', 'stability'), ('Sólo figuras', 'figures'), ('Sólo cálculos', 'calculations'), ('Tablas', 'tables'), ('Información', 'information'), ('DCAE / COSMOS', 'frontier'), ('Grafo y TikZ', 'graph')])
button=widgets.Button(description='Ejecutar')
output=widgets.Output()
def execute_block(_):
    button.disabled=True
    try:
        with output:
            output.clear_output()
            result=subprocess.run([sys.executable,str(WORK/'run.py'),selector.value],cwd=WORK,capture_output=True,text=True)
            print(result.stdout)
            print(result.stderr)
            result.check_returncode()
    finally:button.disabled=False
button.on_click(execute_block)
display(widgets.VBox([selector,button,output]))

## 4 · Guardar y descargar
El ZIP conserva resultados, fuentes y registros. Activá DESCARGAR en Colab para descargarlo.

In [ ]:
import shutil
archive=shutil.make_archive(str(WORK.parent/(WORK.name+'_outputs')),'zip',WORK)
print(archive)
DOWNLOAD=False
if DOWNLOAD:
    from google.colab import files
    files.download(archive)